# AIC HCMC — CapRL-Video-4B caption case studies

**Before running: `Kernel → Restart & Clear Outputs`**

Spot-checks [`internlm/CapRL-Video-4B`](https://huggingface.co/internlm/CapRL-Video-4B) caption quality on a curated set of known-hard shots. This uses the model's **own documented defaults** (prompt, vLLM load, sampling params) from the [model card](https://huggingface.co/internlm/CapRL-Video-4B) and [GitHub repo](https://github.com/InternLM/CapRL) — not this repo's production-tuned config in `scripts/captioning/caption_shots.py`. It's a QA tool, not a benchmark — it captions one shot at a time and prints/saves the result for manual review. See `docs/reports/ocr-shot-audit.md` for why several of the seed shots below are known-hard cases.

**Kaggle settings required:**
- Accelerator: any GPU works — the model is loaded single-GPU (`tensor_parallel_size=1`), matching the docs' own `vllm serve ... --tensor-parallel-size=1` example
- Internet: **On** (needed for `pip install` and the CapRL-Video-4B download — the datasets below are read locally once attached, no network needed for those)
- **+ Add Input** the datasets in the table below before running — this notebook reads them directly from `/kaggle/input`, it never calls `kagglehub.dataset_download()` or fetches them over the network

| Content | Kaggle dataset |
| --- | --- |
| 2026 videos (`L` groups) | `tommr211/aic-batch1-videos` |
| 2025 videos (`K` groups) | `taiphamnguyenanh/aic-batch2-videos-part{1,2,3}` |
| Shot boundaries (both years) | `tommr211/aic-shot-intervals` |

Output: `/kaggle/working/caprl_case_study_captions.json`

## 1. Case studies — edit this list to add more shots

In [ ]:
# (video_id, shot_index) pairs to caption. Year/group/dataset are all derived
# automatically from video_id below — just append a tuple here to add a new
# case study later, no other cell needs to change.
CASE_STUDIES = [
    ("L27_V010", 83),
    ("L30_V078", 20),
    ("L30_V078", 21),
    ("L30_V072", 19),
    ("K17_V003", 102),
    ("K19_V016", 158),
    ("L23_V013", 20),
    # Add more (video_id, shot_index) pairs here.
]

## 2. Install pinned dependencies

Same versions as `requirements-captioning.txt` (`pyarrow`, `qwen-vl-utils`, `transformers`, `vllm`), except `decord` is swapped for `torchcodec`: `decord==0.6.0` fails to open valid MP4s on some current Kaggle images, and qwen-vl-utils' torchvision fallback no longer works either since torchvision removed `io.read_video`. `torchcodec` is the maintained replacement both projects point to. See section 3 for the matching `FORCE_QWENVL_VIDEO_READER` setting.

In [ ]:
import subprocess
import sys


def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])


# decord==0.6.0 fails to demux valid h264/aac MP4s on some current Kaggle images
# ("cannot find video stream"), and qwen-vl-utils' torchvision fallback no longer
# works either (torchvision >=0.24 removed io.read_video). torchcodec is the
# actively maintained replacement both projects point to.
pip("pyarrow==25.0.1", "qwen-vl-utils==0.0.14",
    "transformers==5.15.1", "vllm==0.28.0", "torchcodec")

import torch

print("torch:", torch.__version__, "| cuda:", torch.version.cuda,
      "| gpu count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(" ", i, torch.cuda.get_device_name(i))

## 3. Environment

`FORCE_QWENVL_VIDEO_READER=torchcodec` is a Kaggle-environment fix, not a CapRL tuning knob: `decord==0.6.0` fails to open valid MP4s on the current Kaggle image, and qwen-vl-utils' torchvision fallback no longer works either since torchvision removed `io.read_video`. `torchcodec` is the maintained replacement both projects point to.

In [ ]:
import os

os.environ["FORCE_QWENVL_VIDEO_READER"] = "torchcodec"
os.environ["HF_HOME"] = "/tmp/huggingface"

## 4. Resolve videos

Searches `/kaggle/input` directly for each needed video's `.mp4` — no `kagglehub` call, no network. Make sure the datasets above are attached via **+ Add Input** first, or this raises a clear error naming the missing videos.

In [ ]:
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")


def group_of(video_id):
    return video_id[:3]


def year_of(group):
    return "2026" if group.startswith("L") else "2025"


needed_video_ids = sorted({video_id for video_id, _ in CASE_STUDIES})
VIDEO_PATHS = {}
for video_id in needed_video_ids:
    matches = list(INPUT_ROOT.rglob(f"{video_id}.mp4"))
    if matches:
        VIDEO_PATHS[video_id] = matches[0]

missing = [video_id for video_id in needed_video_ids if video_id not in VIDEO_PATHS]
if missing:
    raise FileNotFoundError(
        f"videos not found under {INPUT_ROOT}: {missing} — attach the right "
        "dataset(s) via + Add Input (see the table at the top of this notebook)"
    )
print(f"resolved paths for {len(needed_video_ids)} needed videos")

## 5. Resolve shot boundaries

Same approach: search `/kaggle/input` directly for each video's shot-interval JSON (from `tommr211/aic-shot-intervals`).

In [ ]:
import json

SHOT_DATA = {}
for video_id in needed_video_ids:
    matches = list(INPUT_ROOT.rglob(f"{video_id}.json"))
    if not matches:
        raise FileNotFoundError(
            f"no shot-interval JSON for {video_id} under {INPUT_ROOT} — attach "
            "tommr211/aic-shot-intervals via + Add Input"
        )
    SHOT_DATA[video_id] = json.loads(matches[0].read_text())

records = []
for video_id, shot_index in CASE_STUDIES:
    group = group_of(video_id)
    data = SHOT_DATA[video_id]
    fps = float(data["fps"])
    shot_bounds = data["shots"]
    if not 0 <= shot_index < len(shot_bounds):
        raise IndexError(
            f"{video_id} has {len(shot_bounds)} shots; shot_index {shot_index} out of range"
        )
    start_frame, end_frame = (int(value) for value in shot_bounds[shot_index])
    records.append({
        "year": year_of(group),
        "group": group,
        "video_id": video_id,
        "shot_index": shot_index,
        "start_frame": start_frame,
        "end_frame": end_frame,
        "fps": fps,
        "video_path": str(VIDEO_PATHS[video_id]),
        "start_sec": start_frame / fps,
        "end_sec": (end_frame + 1) / fps,
        "duration_sec": (end_frame - start_frame + 1) / fps,
    })

## 6. Review before spending GPU time

Catch a wrong `shot_index` here, not after a multi-minute model load.

In [ ]:
import pandas as pd

pd.set_option("display.max_colwidth", None)
pd.DataFrame(records)[[
    "video_id", "shot_index", "start_frame", "end_frame", "duration_sec", "video_path",
]]

## 7. Prompt and vLLM input builder

Uses the exact default caption prompt from the [CapRL-Video-4B model card](https://huggingface.co/internlm/CapRL-Video-4B) and [GitHub repo](https://github.com/InternLM/CapRL) — `"Please describe this video in detail."` — with no post-processing (no timestamp stripping, no custom instructions). `video_start`/`video_end` clip to the shot's own time range; that's inherent to captioning one shot rather than a whole video, not a tuning choice.

In [ ]:
PROMPT = "Please describe this video in detail."


def prepare_vllm_input(record, processor):
    from qwen_vl_utils import process_vision_info

    messages = [{
        "role": "user",
        "content": [
            {
                "type": "video", "video": record["video_path"],
                "video_start": record["start_sec"],
                "video_end": record["end_sec"],
            },
            {"type": "text", "text": PROMPT},
        ],
    }]
    prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs, video_kwargs = process_vision_info(
        messages,
        image_patch_size=processor.image_processor.patch_size,
        return_video_kwargs=True,
        return_video_metadata=True,
    )
    multimodal = {}
    if image_inputs is not None:
        multimodal["image"] = image_inputs
    if video_inputs is not None:
        multimodal["video"] = video_inputs
    return {
        "prompt": prompt,
        "multi_modal_data": multimodal,
        "mm_processor_kwargs": video_kwargs,
    }

## 8. Load CapRL-Video-4B (vLLM, documented defaults)

`tensor_parallel_size=1` and `trust_remote_code=True` match the model card's own `vllm serve "/PATH/CapRL-Video-4B" --trust-remote-code --tensor-parallel-size=1` example. `temperature=1.0`, `top_p=1.0`, `repetition_penalty=1.0` match the model card's OpenAI client example exactly.

`max_tokens` has no documented default anywhere — the HF model card, `generation_config.json`, and the GitHub README all leave it as an undefined `max_tokens=max_tokens` placeholder in that same example. `1024` fills in that placeholder by hand; vLLM's own fallback of 16 (if left unset) cuts every caption off right after the opening sentence (confirmed empirically).

`max_model_len` is also a hardware constraint rather than a tuning choice: with no cap, vLLM defaults to the model's native 262,144-token context, which needs ~36 GiB of KV cache — far more than a single T4 (~14.5 GiB) has. vLLM's own error message reports the maximum feasible length on this GPU is ~18,096 tokens at default `gpu_memory_utilization`; `16384` stays safely under that.

In [ ]:
from transformers import AutoProcessor
from vllm import LLM, SamplingParams

MODEL_ID = "internlm/CapRL-Video-4B"

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
engine = LLM(
    model=MODEL_ID,
    trust_remote_code=True,
    tensor_parallel_size=1,
    max_model_len=16_384,  # hardware cap — see markdown above, not a tuning choice
)
# temperature/top_p/repetition_penalty match the model card's own OpenAI
# client example exactly. max_tokens=1024 fills in that example's undefined
# `max_tokens=max_tokens` placeholder (see markdown above for why).
sampling = SamplingParams(
    temperature=1.0, top_p=1.0, repetition_penalty=1.0, max_tokens=1024,
)
print("CapRL-Video-4B ready")

## 9. Caption each shot

One shot at a time (batch size 1) — this is manual QA, not a throughput benchmark, so each caption prints as soon as it's ready.

In [ ]:
rows = []
for record in records:
    model_input = prepare_vllm_input(record, processor)
    output = engine.generate([model_input], sampling_params=sampling, use_tqdm=False)[0]
    completion = output.outputs[0]
    caption = completion.text.strip()
    rows.append({
        **{key: record[key] for key in (
            "year", "group", "video_id", "shot_index", "start_frame", "end_frame",
            "start_sec", "end_sec", "duration_sec",
        )},
        "caption": caption,
        "model_id": MODEL_ID,
        "output_tokens": len(completion.token_ids),
    })
    print(f"--- {record['video_id']} shot {record['shot_index']} "
          f"({record['duration_sec']:.1f}s) ---")
    print(caption, "\n")

## 10. Review results

In [ ]:
results = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", None)
results

## 11. Save

In [ ]:
OUTPUT = Path("/kaggle/working/caprl_case_study_captions.json")
temporary = OUTPUT.with_suffix(".tmp")
temporary.write_text(json.dumps(rows, indent=2, ensure_ascii=False) + "\n")
temporary.replace(OUTPUT)
print("saved:", OUTPUT)

## Adding more case studies later

1. Add `(video_id, shot_index)` tuples to `CASE_STUDIES` in section 1.
2. If the new video's dataset isn't already attached, add it via **+ Add Input**
   (see the table at the top for which dataset covers which group).
3. Re-run from **section 4** onward — sections 2-3 (dependency install, env
   vars) don't need to be re-run once the kernel already has them.

Good sources for more hard shots: the reviewer shot-exclusion workflow in
`docs/shot-exclusions.md`, and the clip-export dashboard in
`docs/clip-export.md`.

This notebook intentionally uses CapRL-Video-4B's own documented defaults
(prompt, sampling params, single-GPU load), not this repo's production-tuned
config in `scripts/captioning/caption_shots.py` — the two are not meant to be
directly comparable.